In [1]:
import os
import shutil

# This force-deletes everything in the working directory to free up that 19.5GB
for filename in os.listdir('/kaggle/working'):
    file_path = os.path.join('/kaggle/working', filename)
    try:
        if os.path.isfile(file_path) or os.path.islink(file_path):
            os.unlink(file_path)
        elif os.path.isdir(file_path):
            shutil.rmtree(file_path)
    except Exception as e:
        print(f'Failed to delete {file_path}. Reason: {e}')

print("🧹 Disk space cleared! You should see the 19.5GiB drop to 0 shortly.")

🧹 Disk space cleared! You should see the 19.5GiB drop to 0 shortly.


In [2]:
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

In [3]:
import os
# This will list the main folders for your datasets
for dirname, _, _ in os.walk('/kaggle/input'):
    print(dirname)

/kaggle/input
/kaggle/input/datasets
/kaggle/input/datasets/vipoooool
/kaggle/input/datasets/vipoooool/new-plant-diseases-dataset
/kaggle/input/datasets/vipoooool/new-plant-diseases-dataset/New Plant Diseases Dataset(Augmented)
/kaggle/input/datasets/vipoooool/new-plant-diseases-dataset/New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)
/kaggle/input/datasets/vipoooool/new-plant-diseases-dataset/New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)/valid
/kaggle/input/datasets/vipoooool/new-plant-diseases-dataset/New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)/valid/Tomato___Late_blight
/kaggle/input/datasets/vipoooool/new-plant-diseases-dataset/New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)/valid/Tomato___healthy
/kaggle/input/datasets/vipoooool/new-plant-diseases-dataset/New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)/valid/Grape___healthy
/kaggle/input/

In [4]:
import shutil
import os
if os.path.exists('/kaggle/working/unified_dataset'):
    shutil.rmtree('/kaggle/working/unified_dataset')
print("🧹 Workspace cleaned!")

🧹 Workspace cleaned!


In [5]:
import pandas as pd
import os

# 1. Scan all folders to create a list of paths
all_data = []
root_input = '/kaggle/input'

for root, dirs, files in os.walk(root_input):
    images = [f for f in files if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    if images:
        # Get the class name from the folder name
        class_name = os.path.basename(root)
        for img in images:
            all_data.append({
                'filepath': os.path.join(root, img),
                'label': class_name  # Ensure you are capturing the label!
            })

# 2. Convert to a DataFrame (This is your "Virtual Dataset")
df = pd.DataFrame(all_data)
print(f"✅ Found {len(df)} images across {df['label'].nunique()} classes.")

# 3. Create the Data Generators using the DataFrame
from tensorflow.keras.preprocessing.image import ImageDataGenerator

datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    rotation_range=20,
    horizontal_flip=True
)

train_gen = datagen.flow_from_dataframe(
    dataframe=df,
    x_col='filepath',
    y_col='label',
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset='training'
)

val_gen = datagen.flow_from_dataframe(
    dataframe=df,
    x_col='filepath',
    y_col='label',
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset='validation'
)

✅ Found 235615 images across 147 classes.


2026-04-21 11:20:14.179017: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776770414.407428      22 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776770414.470061      22 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776770414.978863      22 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776770414.978911      22 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776770414.978914      22 computation_placer.cc:177] computation placer alr

Found 188492 validated image filenames belonging to 147 classes.
Found 47123 validated image filenames belonging to 147 classes.


In [6]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models

# 1. Load the pre-trained MobileNetV2 base
base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False  # Keep the pre-trained knowledge frozen

# 2. Build the "Head" of the model
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.5),  # This prevents the "random guessing" you saw earlier
    layers.Dense(171, activation='softmax') # 171 matches your found classes
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001), # Lower rate = Higher precision
    loss='categorical_crossentropy', 
    metrics=['accuracy']
)

model.summary()

I0000 00:00:1776771260.440825      22 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1776771260.446740      22 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 171)            │       219,051 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,477,035 (9.45 MB)

 Trainable params: 219,051 (855.67 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [7]:
import pandas as pd
import os
from tensorflow.keras.preprocessing.image import ImageDataGenerator

all_data = []
# This will scan whatever is left in your input folder
for root, dirs, files in os.walk('/kaggle/input'):
    images = [f for f in files if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    if images:
        class_name = os.path.basename(root)
        for img in images:
            all_data.append({'filepath': os.path.join(root, img), 'label': class_name})

df = pd.DataFrame(all_data)
datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)

# We use batch_size=32 to save memory
train_gen = datagen.flow_from_dataframe(
    dataframe=df, x_col='filepath', y_col='label',
    target_size=(224, 224), batch_size=32, class_mode='categorical', subset='training'
)

val_gen = datagen.flow_from_dataframe(
    dataframe=df, x_col='filepath', y_col='label',
    target_size=(224, 224), batch_size=32, class_mode='categorical', subset='validation'
)

Found 188492 validated image filenames belonging to 147 classes.
Found 47123 validated image filenames belonging to 147 classes.


In [8]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models
import json

# Define Model
base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False 

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(512, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(len(train_gen.class_indices), activation='softmax') # Dynamic class count
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Train for 1 Epoch only for the presentation (to ensure it finishes NOW)
model.fit(train_gen, validation_data=val_gen, epochs=1)

# Save
model.save('agriscience_final_model.h5')
with open('class_indices.json', 'w') as f:
    json.dump(train_gen.class_indices, f)

print("✅ Model ready for presentation!")

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()
I0000 00:00:1776772325.149249      69 service.cc:152] XLA service 0x7ff6b810e430 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1776772325.149312      69 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1776772325.149320      69 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1776772326.276142      69 cuda_dnn.cc:529] Loaded cuDNN version 91002
2026-04-21 11:52:15.413147: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal

 234/5891 ━━━━━━━━━━━━━━━━━━━━ 31:25 333ms/step - accuracy: 0.3051 - loss: 3.1143

/usr/local/lib/python3.12/dist-packages/PIL/TiffImagePlugin.py:950: UserWarning: Corrupt EXIF data.  Expecting to read 4 bytes but only got 0. 
  warnings.warn(str(msg))


1154/5891 ━━━━━━━━━━━━━━━━━━━━ 27:39 350ms/step - accuracy: 0.5110 - loss: 1.9520

/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


...

2026-04-21 12:09:59.457679: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-04-21 12:09:59.609375: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-04-21 12:09:59.745391: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


5891/5891 ━━━━━━━━━━━━━━━━━━━━ 0s 337ms/step - accuracy: 0.6486 - loss: 1.2307

2026-04-21 12:31:17.513660: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-04-21 12:31:17.649803: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


5891/5891 ━━━━━━━━━━━━━━━━━━━━ 2360s 398ms/step - accuracy: 0.6486 - loss: 1.2306 - val_accuracy: 0.8569 - val_loss: 0.3539


✅ Model ready for presentation!
